# Paso 1 · Rasgos por época + XGBoost con canales enmascarados

**Proyecto de grado — Maestría en Inteligencia Artificial (MAIA), Universidad de los Andes · Grupo 3**

Clasificación automática de los estadios del sueño (W, N1, N2, N3, REM) en épocas de 30 s, con cualquier combinación de canales que incluya al menos un EEG: Fpz-Cz, Pz-Oz o ambos, con o sin EOG y con o sin EMG. Son **12 combinaciones**.

Este es el primero de los dos modelos base del sistema. Cada época se describe con **719 rasgos** calculados a partir de la señal, y un único XGBoost aprende a clasificarlas. Para que el mismo modelo sirva con cualquier combinación de canales, en el entrenamiento se "apagan" canales al azar: sus columnas se ponen en NaN.

| Sección | Contenido |
|---|---|
| 1 | Configuración y partición |
| 2 | Lectura de las noches |
| 3 | Rasgos base (espectro, forma de la señal, EOG, EMG) |
| 4 | Ondículas y pendiente 1/f |
| 5 | Tabla final con contexto entre épocas |
| 6 | Enmascaramiento de canales |
| 7 | Entrenamiento: validación cruzada y modelo final |
| 8 | Resultados por combinación |
| 9 | Matrices de confusión |

Las probabilidades que produce este paso (fuera de fold y de test) son la entrada del meta-modelo del paso 3.

## 1. Configuración

Los datos van en `data/`, en la raíz del repositorio:

- `data/sleep-cassette/`: los 306 archivos EDF de Sleep Cassette (PSG + hipnograma de 153 noches);
- `data/SC-subjects.xls`: metadatos por noche (aquí se usa la hora en que se apagó la luz).

Los rasgos se guardan en `data/procesados/paso_1/`. Si ya existen, se leen.

In [ ]:
import json
import time
import warnings
from pathlib import Path

import antropy as ant
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import pywt
import xgboost as xgb
from joblib import Parallel, delayed
from scipy import signal, stats
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# Raíz del repositorio: la carpeta que tiene particiones.json
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "particiones.json").exists())
DATA = RAIZ / "data"
EDF = DATA / "sleep-cassette"
SUJETOS_XLS = DATA / "SC-subjects.xls"
PROCESADOS = DATA / "procesados" / "paso_1"
SALIDA = RAIZ / "paso_3_meta_model" / "entradas" / "paso_1_xgboost.npz"
PROCESADOS.mkdir(parents=True, exist_ok=True)
SALIDA.parent.mkdir(parents=True, exist_ok=True)

SEED = 42
N_PROCESOS = 8      # procesos para calcular los rasgos
PRUEBA = False      # True: 1 fold y 50 rondas, solo para comprobar que todo corre

particion = json.loads((RAIZ / "particiones.json").read_text())
CONFIGS = {k: tuple(v) for k, v in particion["combinaciones"].items()}
ESTADIOS = ["W", "N1", "N2", "N3", "REM"]
print(f"{len(particion['sujetos_test'])} sujetos de test, {len(particion['sujetos_desarrollo'])} de desarrollo, "
      f"{particion['n_folds']} folds | {len(CONFIGS)} combinaciones de canales")

## 2. Lectura de las noches

Cada noche `SC4ssN` (ss = sujeto, N = noche) tiene un archivo de señales (`*-PSG.edf`) y uno de anotaciones (`*-Hypnogram.edf`). Las anotaciones vienen en la norma R&K, con inicio y duración en segundos. Se pasan a una etiqueta por época de 30 s y se convierten a la norma AASM: S3 y S4 se juntan en N3.

Las épocas `Sleep stage ?` y `Movement time` no tienen estadio y quedan con etiqueta −1.

Cada registro dura ~22 h y casi todo es vigilia. Se conserva la **ventana de sueño**: desde 30 min antes del primer estadio de sueño hasta 30 min después del último.

In [ ]:
STAGE_MAP = {"Sleep stage W": 0, "Sleep stage 1": 1, "Sleep stage 2": 2,
             "Sleep stage 3": 3, "Sleep stage 4": 3, "Sleep stage R": 4}
UNKNOWN = -1
FS = 100      # Hz de EEG y EOG
EPOCH = 30    # segundos por época


def list_nights(raw_dir):
    """[(noche, ruta_psg, ruta_hipnograma)] ordenado por noche."""
    psgs = {p.name[:6]: p for p in Path(raw_dir).glob("SC4*-PSG.edf")}
    hyps = {p.name[:6]: p for p in Path(raw_dir).glob("SC4*-Hypnogram.edf")}
    return [(k, psgs[k], hyps[k]) for k in sorted(psgs) if k in hyps]


def hypnogram_labels(hyp_path, n_epochs, epoch_sec=EPOCH):
    """Etiqueta (0..4 o -1) de cada época del registro, contando desde su inicio."""
    ann = mne.read_annotations(hyp_path)
    labels = np.full(n_epochs, UNKNOWN, dtype=np.int64)
    for onset, duration, desc in zip(ann.onset, ann.duration, ann.description):
        start = int(round(onset / epoch_sec))
        stop = min(start + int(round(duration / epoch_sec)), n_epochs)
        labels[start:stop] = STAGE_MAP.get(desc, UNKNOWN)
    return labels


def sleep_window(labels, wake_margin_min=30, epoch_sec=EPOCH):
    """(primera, última) época del periodo de sueño, con 30 min de vigilia a cada lado."""
    sleep_idx = np.where(labels > 0)[0]
    margin = int(wake_margin_min * 60 / epoch_sec)
    return max(sleep_idx[0] - margin, 0), min(sleep_idx[-1] + margin, len(labels) - 1)


nights = list_nights(EDF)
print(f"{len(nights)} noches en {EDF}")

## 3. Rasgos base

Todos los rasgos se calculan sobre la ventana de sueño completa, **incluidas las épocas sin etiqueta**, para no romper la continuidad cuando después se miren las épocas vecinas. El prefijo de cada columna dice de qué canal sale (`fpz_`, `pz_`, `eog_`, `emg_`, `time_`), y eso es lo que permite apagar un canal completo.

**Por EEG (36 rasgos):**
- Welch con ventanas de 4 s: potencia absoluta y relativa en delta lenta (0.5–1 Hz), delta (1–4), theta (4–8), alfa (8–12), sigma (12–16) y beta (16–30); potencia total, cocientes, SEF95, frecuencia pico y entropía espectral.
- Forma de la señal: desviación, IQR, asimetría, curtosis, Hjorth, cruces por cero, longitud de línea, entropía de permutación y dimensión de Higuchi.
- 6 sub-ventanas de 5 s: máximo y variación de la potencia sigma y delta, y de la amplitud pico a pico. Husos y complejos K duran menos de 2 s, así que se notan mejor en ventanas cortas.

**EOG (8):** potencia de movimientos lentos (0.3–2 Hz) y rápidos (2–5 Hz), amplitud y la correlación con Fpz-Cz, que es frontal y capta parte del movimiento ocular. Esta última **depende de Fpz-Cz** aunque su prefijo sea `eog_`, y se apaga también cuando falta Fpz-Cz (sección 6).

**EMG (6):** el EMG viene a 1 Hz (es una envolvente: 30 valores por época), así que solo se resumen esos 30 valores.

**Hora:** horas desde que se apagó la luz. N3 se concentra al inicio de la noche y REM al final.

In [ ]:
CHANNELS = {"fpz": "EEG Fpz-Cz", "pz": "EEG Pz-Oz", "eog": "EOG horizontal", "emg": "EMG submental"}
EEG_BANDS = {"sdelta": (0.5, 1), "delta": (1, 4), "theta": (4, 8), "alpha": (8, 12), "sigma": (12, 16), "beta": (16, 30)}
EPS = 1e-12


def _band_power(f, psd, lo, hi):
    idx = (f >= lo) & (f < hi)
    return psd[:, idx].sum(1) * (f[1] - f[0])


def eeg_features(x, p):
    """x: (n_épocas, 3000) en µV. p: prefijo del canal."""
    out = {}
    f, psd = signal.welch(x, fs=FS, nperseg=4 * FS, noverlap=2 * FS, axis=-1)
    total = _band_power(f, psd, 0.5, 30) + EPS
    bp = {b: _band_power(f, psd, lo, hi) + EPS for b, (lo, hi) in EEG_BANDS.items()}
    for b, v in bp.items():
        out[f"{p}_abs_{b}"] = np.log10(v)
        out[f"{p}_rel_{b}"] = v / total
    out[f"{p}_abs_total"] = np.log10(total)
    out[f"{p}_ratio_delta_theta"] = np.log10(bp["delta"] / bp["theta"])
    out[f"{p}_ratio_delta_beta"] = np.log10(bp["delta"] / bp["beta"])
    out[f"{p}_ratio_theta_alpha"] = np.log10(bp["theta"] / bp["alpha"])
    out[f"{p}_ratio_slow_fast"] = np.log10((bp["delta"] + bp["theta"]) / (bp["alpha"] + bp["beta"]))

    band = (f >= 0.5) & (f < 30)
    fb, pb = f[band], psd[:, band] + EPS
    cum = np.cumsum(pb, 1) / pb.sum(1, keepdims=True)
    out[f"{p}_sef95"] = fb[np.argmax(cum >= 0.95, axis=1)]
    out[f"{p}_peak_freq"] = fb[np.argmax(pb, 1)]
    pn = pb / pb.sum(1, keepdims=True)
    out[f"{p}_spec_entropy"] = -(pn * np.log2(pn)).sum(1) / np.log2(pn.shape[1])

    out[f"{p}_std"] = x.std(1)
    q75, q25 = np.percentile(x, [75, 25], axis=1)
    out[f"{p}_iqr"] = q75 - q25
    out[f"{p}_skew"] = stats.skew(x, axis=1)
    out[f"{p}_kurt"] = stats.kurtosis(x, axis=1)
    mob, comp = ant.hjorth_params(x, axis=1)
    out[f"{p}_hjorth_mob"], out[f"{p}_hjorth_comp"] = mob, comp
    out[f"{p}_zcr"] = ant.num_zerocross(x, normalize=True, axis=1)
    out[f"{p}_line_length"] = np.abs(np.diff(x, axis=1)).mean(1)
    out[f"{p}_perm_entropy"] = np.array([ant.perm_entropy(v, order=3, normalize=True) for v in x])
    out[f"{p}_higuchi"] = np.array([ant.higuchi_fd(v.astype(np.float64)) for v in x])

    # 6 sub-ventanas de 5 s
    sub = x.reshape(len(x), 6, 5 * FS)
    fs_, psub = signal.welch(sub, fs=FS, nperseg=250, noverlap=125, axis=-1)
    for b, (lo, hi) in [("sigma", (12, 16)), ("delta", (0.5, 4))]:
        idx = (fs_ >= lo) & (fs_ < hi)
        lp = np.log10(psub[..., idx].sum(-1) * (fs_[1] - fs_[0]) + EPS)
        out[f"{p}_sub_{b}_max"] = lp.max(1)
        out[f"{p}_sub_{b}_std"] = lp.std(1)
    ptp = np.ptp(sub, axis=-1)
    out[f"{p}_sub_ptp_max"] = ptp.max(1)
    out[f"{p}_sub_ptp_std"] = ptp.std(1)
    return out


def eog_features(x, fpz=None):
    out = {}
    f, psd = signal.welch(x, fs=FS, nperseg=4 * FS, noverlap=2 * FS, axis=-1)
    out["eog_abs_slow"] = np.log10(_band_power(f, psd, 0.3, 2) + EPS)
    out["eog_abs_rapid"] = np.log10(_band_power(f, psd, 2, 5) + EPS)
    out["eog_abs_total"] = np.log10(_band_power(f, psd, 0.3, 30) + EPS)
    out["eog_std"] = x.std(1)
    out["eog_line_length"] = np.abs(np.diff(x, axis=1)).mean(1)
    out["eog_max_abs"] = np.abs(x).max(1)
    out["eog_kurt"] = stats.kurtosis(x, axis=1)
    if fpz is not None:
        xc = x - x.mean(1, keepdims=True)
        fc = fpz - fpz.mean(1, keepdims=True)
        out["eog_corr_fpz"] = (xc * fc).sum(1) / (np.sqrt((xc**2).sum(1) * (fc**2).sum(1)) + EPS)
    return out


def emg_features(v):
    """v: (n_épocas, 30) valores a 1 Hz de la envolvente del EMG, en µV."""
    return {"emg_mean": v.mean(1), "emg_median": np.median(v, 1), "emg_std": v.std(1),
            "emg_min": v.min(1), "emg_max": v.max(1), "emg_log_mean": np.log10(np.abs(v.mean(1)) + EPS)}


def lights_off_offset(meas_date, lights_off):
    """Segundos desde el inicio del registro hasta que se apagó la luz."""
    start = meas_date.hour * 3600 + meas_date.minute * 60 + meas_date.second
    lo = lights_off.hour * 3600 + lights_off.minute * 60 + lights_off.second
    delta = lo - start
    return delta + 86400 if delta < 0 else delta


def extract_night(job):
    """Rasgos base de una noche. job = (noche, psg, hipnograma, hora de apagado de la luz)."""
    nid, psg, hyp, lights_off = job
    warnings.filterwarnings("ignore")
    raw = mne.io.read_raw_edf(psg, include=list(CHANNELS.values()), preload=True, verbose="error")
    n_epochs = int(raw.n_times // (FS * EPOCH))
    labels = hypnogram_labels(hyp, n_epochs, EPOCH)
    first, last = sleep_window(labels)
    idx = np.arange(first, last + 1)

    data = {k: raw.get_data(picks=ch)[0] * 1e6 for k, ch in CHANNELS.items()}  # V -> µV
    cut = lambda s: s[: n_epochs * FS * EPOCH].reshape(n_epochs, FS * EPOCH)[idx]  # noqa: E731
    fpz, pz, eog = cut(data["fpz"]), cut(data["pz"]), cut(data["eog"])
    # MNE interpola el EMG (1 Hz) a 100 Hz: cada 100 muestras está el valor original
    emg = data["emg"][: n_epochs * FS * EPOCH : FS].reshape(n_epochs, EPOCH)[idx]

    feats = {**eeg_features(fpz, "fpz"), **eeg_features(pz, "pz"), **eog_features(eog, fpz), **emg_features(emg)}
    offset = lights_off_offset(raw.info["meas_date"], lights_off)
    df = pd.DataFrame(feats).astype(np.float32)
    df.insert(0, "time_hours_since_lights_off", ((idx * EPOCH - offset) / 3600).astype(np.float32))
    df.insert(0, "y", labels[idx])
    df.insert(0, "epoch_idx", idx)
    df.insert(0, "subject", int(nid[3:5]))
    df.insert(0, "night", nid)
    return df

Las 153 noches se procesan en paralelo (unos 15 min con 8 procesos). El resultado se guarda y, si ya existe, se lee directamente.

In [ ]:
RASGOS_BASE = PROCESADOS / "rasgos_base.parquet"
if not RASGOS_BASE.exists():
    t0 = time.time()
    meta = pd.read_excel(SUJETOS_XLS)
    lights = {(int(r.subject), int(r.night)): r.LightsOff for _, r in meta.iterrows()}
    jobs = [(n, p, h, lights[(int(n[3:5]), int(n[5]))]) for n, p, h in nights]
    tablas = Parallel(n_jobs=N_PROCESOS)(delayed(extract_night)(j) for j in jobs)
    pd.concat(tablas, ignore_index=True).to_parquet(RASGOS_BASE)
    print(f"rasgos base calculados en {time.time() - t0:.0f} s")

base = pd.read_parquet(RASGOS_BASE)
lab = base[base.y >= 0]
print(f"{base.night.nunique()} noches · {len(base):,} épocas en la ventana · {len(lab):,} con etiqueta")
print("épocas por estadio:", dict(zip(ESTADIOS, lab.y.value_counts().sort_index().tolist())))
assert len(lab) == 195_469

## 4. Ondículas y pendiente 1/f

**Ondículas (DWT).** Cada época de Fpz-Cz, Pz-Oz y EOG se descompone con una ondícula biortogonal 2.2 en 4 niveles. Salen 5 sub-bandas: cA4 ≈ 0–3 Hz, cD4 ≈ 3–6, cD3 ≈ 6–12, cD2 ≈ 12–25 y cD1 ≈ 25–50 Hz. De cada una se calculan 8 medidas: log-energía, energía de Teager, movilidad y complejidad de Hjorth, curtosis, asimetría y dimensiones fractales de Katz y Petrosian.

**Pendiente 1/f.** El espectro del EEG tiene una parte "de fondo" que cae como 1/f, y su pendiente cambia con el nivel de alerta (W, N1, REM). Se ajusta una recta en escala log-log entre 2 y 30 Hz, sin la zona de 7.5–16 Hz, para que el alfa y los husos no la desvíen. Se guardan la pendiente, el intercepto y el R².

In [ ]:
DWT_BANDS = ["cA4", "cD4", "cD3", "cD2", "cD1"]


def dwt_features(x, p):
    out = {}
    coeffs = pywt.wavedec(x, "bior2.2", level=4, axis=-1)
    for band, c in zip(DWT_BANDS, coeffs):
        q = f"{p}_dwt_{band}"
        out[f"{q}_logenergy"] = np.log10((c**2).mean(1) + EPS)
        out[f"{q}_teager"] = np.log10(np.abs(c[:, 1:-1] ** 2 - c[:, :-2] * c[:, 2:]).mean(1) + EPS)
        mob, comp = ant.hjorth_params(c, axis=1)
        out[f"{q}_hjorth_mob"], out[f"{q}_hjorth_comp"] = mob, comp
        out[f"{q}_kurt"] = stats.kurtosis(c, axis=1)
        out[f"{q}_skew"] = stats.skew(c, axis=1)
        out[f"{q}_katz"] = np.array([ant.katz_fd(v) for v in c])
        out[f"{q}_petrosian"] = np.array([ant.petrosian_fd(v) for v in c])
    return out


def aperiodic_features(x, p):
    f, psd = signal.welch(x, fs=FS, nperseg=4 * FS, noverlap=2 * FS, axis=-1)
    m = (f >= 2) & (f <= 30) & ~((f > 7.5) & (f < 16))
    lx = np.log10(f[m])
    ly = np.log10(psd[:, m] + EPS)
    A = np.vstack([lx, np.ones_like(lx)]).T
    coef, *_ = np.linalg.lstsq(A, ly.T, rcond=None)  # una recta por época
    pred = (A @ coef).T
    ss_res = ((ly - pred) ** 2).sum(1)
    ss_tot = ((ly - ly.mean(1, keepdims=True)) ** 2).sum(1) + EPS
    return {f"{p}_ap_slope": coef[0], f"{p}_ap_offset": coef[1], f"{p}_ap_r2": 1 - ss_res / ss_tot}


def extract_night_dwt(job):
    nid, psg, hyp = job
    warnings.filterwarnings("ignore")
    chans = {k: CHANNELS[k] for k in ("fpz", "pz", "eog")}
    raw = mne.io.read_raw_edf(psg, include=list(chans.values()), preload=True, verbose="error")
    n = int(raw.n_times // (FS * EPOCH))
    labels = hypnogram_labels(hyp, n, EPOCH)
    first, last = sleep_window(labels)
    idx = np.arange(first, last + 1)
    ep = {k: (raw.get_data(picks=ch)[0][: n * FS * EPOCH] * 1e6).reshape(n, FS * EPOCH)[idx] for k, ch in chans.items()}
    feats = {}
    for k in ("fpz", "pz", "eog"):
        feats.update(dwt_features(ep[k], k))
    for k in ("fpz", "pz"):
        feats.update(aperiodic_features(ep[k], k))
    df = pd.DataFrame(feats).astype(np.float32)
    df.insert(0, "epoch_idx", idx)
    df.insert(0, "night", nid)
    return df


RASGOS_DWT = PROCESADOS / "rasgos_ondiculas_1f.parquet"
if not RASGOS_DWT.exists():
    t0 = time.time()
    tablas = Parallel(n_jobs=N_PROCESOS)(delayed(extract_night_dwt)(j) for j in nights)
    pd.concat(tablas, ignore_index=True).to_parquet(RASGOS_DWT)
    print(f"ondículas y 1/f calculadas en {time.time() - t0:.0f} s")
dwt = pd.read_parquet(RASGOS_DWT)
print(dwt.shape, "| valores faltantes:", int(dwt.isna().sum().sum()))

## 5. Tabla final con contexto entre épocas

Se unen las dos tablas y se agrega lo que un técnico mira al calificar: **las épocas vecinas**. Todo se calcula dentro de cada noche y en orden:

1. **z-score robusto por noche** (mediana e IQR) de cada rasgo: compara la época con lo normal de esa persona esa noche (columnas `*_z`).
2. **Medias móviles centradas** de 5 y 15 épocas (2.5 y 7.5 min) de los rasgos clave (lista `KEY_SUFFIXES`) (`*_roll5`, `*_roll15`).
3. **Valores de las épocas t−2, t−1, t+1 y t+2** de esos mismos rasgos (`*_lagm2` … `*_lagp2`).

Se usa contexto futuro porque la calificación es offline, sobre la noche completa.

El orden de las columnas importa: XGBoost sortea columnas por posición (`colsample_bytree`), así que con otro orden el modelo sale distinto.

In [ ]:
META_COLS = ["night", "subject", "epoch_idx", "y"]
KEY_SUFFIXES = [
    "abs_sdelta", "abs_delta", "abs_theta", "abs_alpha", "abs_sigma", "abs_beta",
    "rel_delta", "rel_sigma", "ratio_slow_fast", "spec_entropy", "perm_entropy", "sub_sigma_max",
    "abs_slow", "abs_rapid", "std",          # EOG (y std del EEG)
    "mean", "log_mean",                      # EMG
    "ap_slope", "dwt_cD2_logenergy", "dwt_cD3_logenergy", "dwt_cA4_logenergy",
]


def base_feature_cols(df):
    return [c for c in df.columns if c not in META_COLS and not c.startswith("time_")]


def add_context(df, rolls=(5, 15), lags=(-2, -1, 1, 2)):
    df = df.sort_values(["night", "epoch_idx"]).reset_index(drop=True)
    base = base_feature_cols(df)
    g = df.groupby("night", sort=False)[base]
    med = g.transform("median")
    iqr = g.transform(lambda s: s.quantile(0.75) - s.quantile(0.25)).replace(0, 1.0)
    z = ((df[base] - med) / iqr).astype(np.float32)
    z.columns = [f"{c}_z" for c in base]

    keys = [f"{c}_z" for c in base if any(c.endswith("_" + s) for s in KEY_SUFFIXES)]
    gz = z[keys].groupby(df["night"], sort=False)
    parts = [df, z]
    for w in rolls:
        r = gz.rolling(w, center=True, min_periods=1).mean().reset_index(level=0, drop=True).sort_index()
        r.columns = [f"{c}_roll{w}" for c in keys]
        parts.append(r.astype(np.float32))
    for k in lags:
        s = gz.shift(-k)  # lag +1 = la época siguiente
        s.columns = [f"{c}_lag{'p' if k > 0 else 'm'}{abs(k)}" for c in keys]
        parts.append(s.astype(np.float32))
    return pd.concat(parts, axis=1)


feat = base.merge(dwt, on=["night", "epoch_idx"], how="left", validate="one_to_one")
assert not feat.isna().any().any()
full = add_context(feat)

CHANNELS_ALL = ("fpz", "pz", "eog", "emg")
prefixes = tuple(f"{c}_" for c in CHANNELS_ALL) + ("time_",)
cols = [c for c in full.columns if c not in META_COLS and c.startswith(prefixes)]
print(f"tabla: {full.shape[0]:,} épocas × {len(cols)} columnas del modelo")
print("por canal:", {c: sum(x.startswith(c + "_") for x in cols) for c in ("fpz", "pz", "eog", "emg", "time")})

## 6. Enmascaramiento de canales

Para cada canal se arma la lista de columnas que dependen de él. `eog_corr_fpz` queda también en la lista de Fpz-Cz: si falta Fpz-Cz, esa columna no se puede calcular aunque haya EOG.

En el entrenamiento, **a cada época se le sortea una de las 12 combinaciones** y las columnas de los canales ausentes se ponen en NaN. XGBoost aprende en cada división hacia qué rama mandar los valores faltantes, así que un solo modelo sirve para cualquier combinación.

In [ ]:
CROSS = {"corr_fpz": "fpz"}  # rasgos con prefijo de un canal que también usan otro


def channel_index(cols):
    ch = np.array([c.split("_")[0] for c in cols])
    idx = {c: set(np.where(ch == c)[0]) for c in CHANNELS_ALL}
    for i, col in enumerate(cols):
        for key, other in CROSS.items():
            if key in col:
                idx[other].add(i)
    return {c: np.array(sorted(v), dtype=int) for c, v in idx.items()}


def apply_mask(X, configs_per_row, idx):
    """Pone en NaN las columnas de los canales ausentes de cada fila. Modifica X."""
    names = np.asarray(configs_per_row)
    for cfg, chans in CONFIGS.items():
        r = np.where(names == cfg)[0]
        if len(r) == 0:
            continue
        for ch in CHANNELS_ALL:
            if ch not in chans:
                X[np.ix_(r, idx[ch])] = np.nan
    return X


def random_mask(df, cols, idx, rng):
    X = df[cols].to_numpy(np.float32, copy=True)
    return apply_mask(X, rng.choice(list(CONFIGS), size=len(X)), idx)


idx = channel_index(cols)
demo = full[full.y >= 0].head(4)[cols].to_numpy(np.float32, copy=True)
ejemplo = ["fpz", "pz_eog", "eeg2_emg", "fpz_eog_emg"]
apply_mask(demo, ejemplo, idx)
pd.DataFrame({"combinación": ejemplo, **{ch: [f"{np.isnan(r[idx[ch]]).mean():.0%} NaN" for r in demo] for ch in CHANNELS_ALL}})

## 7. Entrenamiento

**Hiperparámetros.** Los eligió una búsqueda con optuna (TPE, 5 folds de desarrollo) en una etapa anterior del proyecto, y aquí no se vuelven a buscar. Los **pesos por clase** son balanceado^α con α = 0.44: compensan el desbalance (N3 es el 6.7 % de las épocas) sin exagerar las clases raras.

**Protocolo** (`particiones.json`):
- **Validación cruzada de 5 folds** sobre los 62 sujetos de desarrollo. En cada fold se entrena con ~44 sujetos, se hace early stopping con los ~5 de validación (también enmascarados) y se predicen los ~13 restantes con las 12 combinaciones. Juntando los 5 folds se obtienen las **predicciones fuera de fold** de las 62 personas.
- **Modelo final** con los 62 sujetos, sin validación, con el promedio de rondas de los folds. Predice el **test de 16 sujetos** con las 12 combinaciones.

Cada sorteo de máscaras tiene su semilla: `[42, fold]` en la CV y `[42, 99]` en el modelo final. Las predicciones incluyen las épocas sin etiqueta (y = −1), porque el meta-modelo mira épocas vecinas. Tarda ~40 min.

In [ ]:
PARAMS = {
    "objective": "multi:softprob", "tree_method": "hist", "eval_metric": "mlogloss", "n_jobs": -1,
    "learning_rate": 0.03394046833983564, "max_depth": 10, "min_child_weight": 43.70990468130503,
    "subsample": 0.9041986740582306, "colsample_bytree": 0.4436910153386966,
    "reg_lambda": 0.019634341572933336, "gamma": 3.4211651325607844,
}
ALPHA = 0.4401524937396013
N_MAX = 50 if PRUEBA else 2000


def class_weights(y, alpha=ALPHA):
    return compute_sample_weight("balanced", y) ** alpha


def rows(subjects, labeled_only=True):
    m = full.subject.isin(list(subjects))
    return full[m & (full.y >= 0)] if labeled_only else full[m]


def fit(train_subj, rng, val_subj=None, n_estimators=N_MAX):
    tr = rows(train_subj)
    p = {**PARAMS, "random_state": SEED}
    Xtr = random_mask(tr, cols, idx, rng)
    if val_subj is None:
        model = xgb.XGBClassifier(**p, n_estimators=n_estimators)
        model.fit(Xtr, tr.y, sample_weight=class_weights(tr.y), verbose=False)
        return model
    va = rows(val_subj)
    Xva = random_mask(va, cols, idx, rng)
    model = xgb.XGBClassifier(**p, n_estimators=n_estimators, early_stopping_rounds=100)
    model.fit(Xtr, tr.y, sample_weight=class_weights(tr.y), eval_set=[(Xva, va.y)],
              sample_weight_eval_set=[class_weights(va.y)], verbose=False)
    return model


def predict_all_configs(model, subjects):
    """combinación -> noche -> (y, probabilidades), con todas las épocas enmascaradas igual."""
    d = rows(subjects, labeled_only=False)
    base_x = d[cols].to_numpy(np.float32)
    groups = d.groupby("night").indices
    out = {}
    for cfg in CONFIGS:
        proba = model.predict_proba(apply_mask(base_x.copy(), [cfg] * len(base_x), idx))
        out[cfg] = {n: (d.y.values[i], proba[i]) for n, i in groups.items()}
    return out


def save_preds(path, preds):
    arr = {}
    for cfg, p in preds.items():
        for n, (y, pr) in p.items():
            arr[f"{cfg}__{n}__y"], arr[f"{cfg}__{n}__pred"] = y, pr
    np.savez_compressed(path, **arr)


def load_preds(path):
    z = np.load(path)
    out = {}
    for k in z.files:
        cfg, n, kind = k.split("__")
        if kind == "y":
            out.setdefault(cfg, {})[n] = (z[k], z[f"{cfg}__{n}__pred"])
    return out

In [ ]:
FOLDS = PROCESADOS / ("folds_prueba" if PRUEBA else "folds")
FOLDS.mkdir(exist_ok=True)
folds = particion["folds"][:1] if PRUEBA else particion["folds"]
t0 = time.time()

rondas = []
for f in folds:
    path, meta = FOLDS / f"fold_{f['fold']}.npz", FOLDS / f"fold_{f['fold']}.json"
    if not meta.exists():  # si se corta, al volver a correr retoma desde el último fold guardado
        rng = np.random.default_rng([SEED, f["fold"]])
        m = fit(f["train"], rng, val_subj=f["val"])
        save_preds(path, predict_all_configs(m, f["test"]))
        meta.write_text(json.dumps({"rondas": m.best_iteration + 1}))
    rondas.append(json.loads(meta.read_text())["rondas"])
    print(f"fold {f['fold'] + 1}: {rondas[-1]} rondas ({time.time() - t0:.0f} s)")

n_final = int(np.mean(rondas))
final_path = FOLDS / "final.npz"
if not final_path.exists():
    final = fit(particion["sujetos_desarrollo"], np.random.default_rng([SEED, 99]), n_estimators=n_final)
    save_preds(final_path, predict_all_configs(final, particion["sujetos_test"]))
print(f"modelo final: {n_final} rondas ({time.time() - t0:.0f} s)")
print("tesis: 592, 656, 660, 508 y 644 rondas por fold; 612 en el modelo final")

oof = {c: {} for c in CONFIGS}
for f in folds:
    for c, p in load_preds(FOLDS / f"fold_{f['fold']}.npz").items():
        oof[c].update(p)
test = load_preds(final_path)

## 8. Resultados por combinación

**Métricas:**
- **F1 macro (MF1):** la principal. Es el promedio del F1 de los 5 estadios con el mismo peso, así que N1, que es el 11 % de las épocas y el más difícil, cuenta tanto como N2.
- **κ de Cohen:** acuerdo con el técnico descontando el azar.
- **F1 de N1.**

Solo se evalúan las épocas con etiqueta. Estas cifras son las del modelo A solo; las del sistema completo están en el paso 3.

In [ ]:
def compute_metrics(y_true, y_pred):
    f1 = f1_score(y_true, y_pred, labels=range(5), average=None, zero_division=0)
    return {"acc": accuracy_score(y_true, y_pred), "mf1": f1.mean(), "kappa": cohen_kappa_score(y_true, y_pred),
            **{f"f1_{s}": v for s, v in zip(ESTADIOS, f1)},
            "confusion": confusion_matrix(y_true, y_pred, labels=range(5))}


def metrics_of(preds):
    y = np.concatenate([y for y, _ in preds.values()])
    p = np.concatenate([p.argmax(1) for _, p in preds.values()])
    return compute_metrics(y[y >= 0], p[y >= 0])


TESIS = {"fpz": (0.769, 0.727), "pz": (0.728, 0.701), "fpz_eog": (0.788, 0.755), "eeg2_eog_emg": (0.801, 0.764)}
filas = []
for c in CONFIGS:
    cv, te = metrics_of(oof[c]), metrics_of(test[c])
    filas.append({"combinación": c, "CV MF1": cv["mf1"], "CV κ": cv["kappa"], "CV F1 N1": cv["f1_N1"],
                  "test MF1": te["mf1"], "test κ": te["kappa"], "test F1 N1": te["f1_N1"],
                  "tesis (CV / test)": "{:.3f} / {:.3f}".format(*TESIS[c]) if c in TESIS else ""})
pd.DataFrame(filas).set_index("combinación").style.format(precision=3)

In [ ]:
# Entrada del paso 3: probabilidades fuera de fold (oof) y de test, por combinación
arr = {}
for c in CONFIGS:
    for key, preds in (("oof", oof[c]), ("test", test[c])):
        for n, (y, p) in preds.items():
            arr[f"{c}__{key}__{n}__y"], arr[f"{c}__{key}__{n}__pred"] = y, p.astype(np.float32)
if not PRUEBA:
    np.savez_compressed(SALIDA, **arr)
    print("guardado:", SALIDA.relative_to(RAIZ))

## 9. Matrices de confusión

Test (16 sujetos), solo con Fpz-Cz y con los 4 canales. Cada fila es el estadio real y suma 100 %.

In [ ]:
def plot_confusion(cm, title, ax):
    cm = np.asarray(cm)
    cmn = cm / cm.sum(1, keepdims=True)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{cmn[i, j]:.0%}\n{cm[i, j]:,}", ha="center", va="center", fontsize=8,
                    color="white" if cmn[i, j] > 0.6 else "black")
    ax.set_xticks(range(5), ESTADIOS)
    ax.set_yticks(range(5), ESTADIOS)
    ax.set_xlabel("predicho")
    ax.set_ylabel("real")
    ax.set_title(title, fontsize=10)


fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, c, nombre in zip(axes, ["fpz", "eeg2_eog_emg"], ["solo Fpz-Cz", "4 canales"]):
    m = metrics_of(test[c])
    plot_confusion(m["confusion"], f"XGBoost · test · {nombre}\nMF1 {m['mf1']:.3f} · κ {m['kappa']:.3f}", ax)
plt.tight_layout()
plt.show()

La confusión más frecuente es N1: se confunde con W y con N2. Es la clase de transición, y en ella el modelo del paso 2, que trabaja sobre la señal cruda, comete errores distintos. Esa diferencia es la que aprovecha el meta-modelo del paso 3.